In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 1: Data Understanding -> File 2 of 5: olist_customers_dataset.csv
==================================================================================
 Purpose : Explore customer location/identity data (used for Tableau dashboard).
           Contains customer IDs mapped to zip code, city, and state.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. DATA COLLECTION
# ----------------------------------------------------------------------------------
section("1. DATA COLLECTION")

DATA_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_customers_dataset.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Shape of dataset -> Rows: {df.shape[0]}, Columns: {df.shape[1]}")


# ----------------------------------------------------------------------------------
# 2. DATASET STRUCTURE
# ----------------------------------------------------------------------------------
section("2. DATASET STRUCTURE (First & Last Records)")
print("\n--- First 5 records ---")
print(df.head())
print("\n--- Last 5 records ---")
print(df.tail())

section("2.1 COLUMN NAMES & DATA TYPES")
print(df.dtypes)

section("2.2 MEMORY USAGE & GENERAL INFO")
df.info(memory_usage="deep")


# ----------------------------------------------------------------------------------
# 3. MISSING VALUE ANALYSIS
# ----------------------------------------------------------------------------------
section("3. MISSING VALUE ANALYSIS")

missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing %", ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_summary)


# ----------------------------------------------------------------------------------
# 4. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("4. DUPLICATE RECORD CHECK")
print(f"Fully duplicated rows              : {df.duplicated().sum()}")
print(f"Duplicated customer_id count       : {df['customer_id'].duplicated().sum()}")
print(f"Duplicated customer_unique_id count: {df['customer_unique_id'].duplicated().sum()}")
print("\nNote: 'customer_id' is unique PER ORDER, while 'customer_unique_id' identifies")
print("the actual person -> repeat customers will share the same customer_unique_id.")


# ----------------------------------------------------------------------------------
# 5. FEATURE SEGREGATION
# ----------------------------------------------------------------------------------
section("5. FEATURE SEGREGATION (Numerical vs Categorical)")

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
for id_col in ["customer_id", "customer_unique_id"]:
    if id_col in categorical_cols:
        categorical_cols.remove(id_col)

print(f"Numerical columns   ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


# ----------------------------------------------------------------------------------
# 6. GEOGRAPHIC DISTRIBUTION (State & City) - key for Tableau maps
# ----------------------------------------------------------------------------------
section("6. GEOGRAPHIC DISTRIBUTION")

print(f"Total unique states : {df['customer_state'].nunique()}")
print(f"Total unique cities : {df['customer_city'].nunique()}")

print("\nTop 10 states by customer count:")
print(df["customer_state"].value_counts().head(10))

print("\nTop 10 cities by customer count:")
print(df["customer_city"].value_counts().head(10))


# ----------------------------------------------------------------------------------
# 7. REPEAT CUSTOMER CHECK (based on customer_unique_id)
# ----------------------------------------------------------------------------------
section("7. REPEAT CUSTOMER ANALYSIS")

repeat_counts = df["customer_unique_id"].value_counts()
repeat_customers = (repeat_counts > 1).sum()
print(f"Unique persons (customer_unique_id): {df['customer_unique_id'].nunique()}")
print(f"Persons appearing more than once   : {repeat_customers}")
print(f"Records tied to a repeat person     : {(repeat_counts[repeat_counts > 1]).sum()}")


# ----------------------------------------------------------------------------------
# 8. CARDINALITY CHECK
# ----------------------------------------------------------------------------------
section("8. CARDINALITY CHECK")
print(df.nunique().sort_values())


# ----------------------------------------------------------------------------------
# 9. INITIAL DATA QUALITY OBSERVATIONS
# ----------------------------------------------------------------------------------
section("9. INITIAL DATA QUALITY OBSERVATIONS")

observations = f"""
1. Dataset contains {df.shape[0]} customer-order records across {df['customer_state'].nunique()} states
   and {df['customer_city'].nunique()} cities.
2. 'customer_id' is unique per order and is the join key to 'olist_orders_dataset.csv'.
3. 'customer_unique_id' identifies the actual person -> use THIS column in Tableau
   when measuring true customer counts, retention, or repeat-purchase behaviour,
   not 'customer_id'.
4. No missing values were found in this file.
5. City names are in free-text lowercase Portuguese and may contain minor spelling
   variants -> consider standardizing/grouping by 'customer_state' for a cleaner
   geographic dashboard view.
"""
print(observations)

section("CUSTOMERS DATASET - DATA UNDERSTANDING COMPLETE")


 1. DATA COLLECTION
Dataset loaded successfully from: /Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_customers_dataset.csv
Shape of dataset -> Rows: 99441, Columns: 5

 2. DATASET STRUCTURE (First & Last Records)

--- First 5 records ---
                        customer_id                customer_unique_id  customer_zip_code_prefix          customer_city customer_state
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0                     14409                 franca             SP
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3                      9790  sao bernardo do campo             SP
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e                      1151              sao paulo             SP
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c                      8775        mogi das cruzes             SP
